In [2]:
for idx in [12, 14, 17]:
    atom = built.GetAtomWithIdx(idx)
    print("built", idx, atom.GetSymbol(), atom.GetFormalCharge(), "5-ring:", atom.IsInRingSize(5), "6-ring:", atom.IsInRingSize(6))

atom = xtal.GetAtomWithIdx(7)
print("crystal", 7, atom.GetSymbol(), atom.GetFormalCharge(), "5-ring:", atom.IsInRingSize(5), "6-ring:", atom.IsInRingSize(6))

built 12 C 0 5-ring: True 6-ring: False
built 14 C 0 5-ring: False 6-ring: True
built 17 N 1 5-ring: False 6-ring: True
crystal 7 C 0 5-ring: True 6-ring: False


In [3]:
from rdkit.Chem import AllChem

smiles_R = "COc1cc2c(cc1OC)C(=O)[C@H](CC1CC[NH+](Cc3ccccc3)CC1)C2"
mol_R = Chem.AddHs(Chem.MolFromSmiles(smiles_R))
print(mol_R.GetNumAtoms())

58


In [4]:
params = AllChem.ETKDGv3()
params.randomSeed = 42
print("embed:", AllChem.EmbedMolecule(mol_R, params))
print("mmff:", AllChem.MMFFOptimizeMolecule(mol_R, maxIters=2000))

embed: 0
mmff: 0


In [5]:
writer = Chem.SDWriter("../prep/donepezil_R_3d.sdf")
writer.write(mol_R)
writer.close()

check = Chem.MolFromMolFile("../prep/donepezil_R_3d.sdf", removeHs=False)
Chem.AssignStereochemistryFrom3D(check)
print(Chem.FindMolChiralCenters(check, includeUnassigned=True, useLegacyImplementation=False))

[(12, 'R'), (14, 'r'), (17, 'R')]


In [6]:
!mk_prepare_ligand.py -i ../prep/donepezil_R_3d.sdf -o ../prep/donepezil_R.pdbqt
!grep -E "TORSDOF|REMARK SMILES " ../prep/donepezil_R.pdbqt

Input molecules processed: 1, skipped: 0
PDBQT files written: 1
PDBQT files not written due to error: 0
Input molecules with errors: 0
REMARK SMILES COc1cc2c(cc1OC)C(=O)[C@H](C[C@H]1CC[N@H+](Cc3ccccc3)CC1)C2
REMARK SMILES IDX 14 1 13 2 3 3 4 4 8 5 5 6 7 7 6 8 28 9 11 10 12 11 2 12 1 13
REMARK SMILES IDX 9 14 10 15 15 16 16 17 27 18 17 19 26 20 18 21 19 23 20 24
REMARK SMILES IDX 21 25 25 26 22 27 24 28 23 29
TORSDOF 6


In [7]:
%%writefile ../prep/vina_config.txt
center_x = -14.108
center_y = -43.833
center_z = 27.670
size_x = 22
size_y = 22
size_z = 22
exhaustiveness = 8
num_modes = 9
seed = 42

Writing ../prep/vina_config.txt


In [8]:
!vina --receptor ../prep/receptor_A.pdbqt --ligand ../prep/donepezil_R.pdbqt --config ../prep/vina_config.txt --out ../results/donepezil_R_redock_out.pdbqt

AutoDock Vina f458505-mod
#################################################################
# If you used AutoDock Vina in your work, please cite:          #
#                                                               #
# J. Eberhardt, D. Santos-Martins, A. F. Tillack, and S. Forli  #
# AutoDock Vina 1.2.0: New Docking Methods, Expanded Force      #
# Field, and Python Bindings, J. Chem. Inf. Model. (2021)       #
# DOI 10.1021/acs.jcim.1c00203                                  #
#                                                               #
# O. Trott, A. J. Olson,                                        #
# AutoDock Vina: improving the speed and accuracy of docking    #
# with a new scoring function, efficient optimization and       #
# multithreading, J. Comp. Chem. (2010)                         #
# DOI 10.1002/jcc.21334                                         #
#                                                               #
# Please see https://github.com/ccsb-scripps/AutoD

In [9]:
from meeko import PDBQTMolecule, RDKitMolCreate
from rdkit.Chem import rdMolAlign
from rdkit.Chem.MolStandardize import rdMolStandardize

pdbqt = PDBQTMolecule.from_file("../results/donepezil_R_redock_out.pdbqt", skip_typing=True)
docked = RDKitMolCreate.from_pdbqt_mol(pdbqt)[0]
docked = rdMolStandardize.Uncharger().uncharge(Chem.RemoveHs(docked))

for conf in docked.GetConformers():
    rmsd = rdMolAlign.CalcRMS(docked, xtal, prbId=conf.GetId())
    print("mode", conf.GetId() + 1, "RMSD", round(rmsd, 2), "Å")


mode 1 RMSD 0.75 Å
mode 2 RMSD 2.48 Å
mode 3 RMSD 9.5 Å
mode 4 RMSD 3.53 Å
mode 5 RMSD 3.19 Å
mode 6 RMSD 4.09 Å
mode 7 RMSD 9.84 Å
mode 8 RMSD 4.78 Å
mode 9 RMSD 9.38 Å


In [10]:
pdbqt_S = PDBQTMolecule.from_file("../results/donepezil_redock_out.pdbqt", skip_typing=True)
docked_S = RDKitMolCreate.from_pdbqt_mol(pdbqt_S)[0]
docked_S = rdMolStandardize.Uncharger().uncharge(Chem.RemoveHs(docked_S))

for conf in docked_S.GetConformers():
    rmsd = rdMolAlign.CalcRMS(docked_S, xtal, prbId=conf.GetId())
    print("S mode", conf.GetId() + 1, "RMSD", round(rmsd, 2), "Å")

S mode 1 RMSD 3.14 Å
S mode 2 RMSD 2.56 Å
S mode 3 RMSD 8.95 Å
S mode 4 RMSD 1.9 Å
S mode 5 RMSD 2.88 Å
S mode 6 RMSD 1.99 Å
S mode 7 RMSD 10.33 Å
S mode 8 RMSD 2.96 Å
S mode 9 RMSD 9.99 Å


In [1]:
%%writefile draw_box.py
from pymol import cmd
from pymol.cgo import BEGIN, LINES, VERTEX, END, COLOR

def draw_box(config="prep/vina_config.txt", name="vina_box"):
    v = {}
    for line in open(config):
        if "=" in line:
            key, value = line.split("=")
            v[key.strip()] = float(value)

    cx, cy, cz = v["center_x"], v["center_y"], v["center_z"]
    hx, hy, hz = v["size_x"] / 2, v["size_y"] / 2, v["size_z"] / 2
    xs, ys, zs = (cx - hx, cx + hx), (cy - hy, cy + hy), (cz - hz, cz + hz)
    corners = [(x, y, z) for x in xs for y in ys for z in zs]

    box = [BEGIN, LINES, COLOR, 1.0, 0.5, 0.0]
    for a in corners:
        for b in corners:
            if a < b and sum(i != j for i, j in zip(a, b)) == 1:
                box += [VERTEX, *a, VERTEX, *b]
    box.append(END)
    cmd.load_cgo(box, name)

cmd.extend("draw_box", draw_box)
draw_box()

Writing draw_box.py
